In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------
# Tables
# -------------------------
bronze_tbl = "capstone.bronze.pharmacies"
geo_tbl = "capstone.silver.geography"
silver_tbl = "capstone.silver.pharmacy"

# -------------------------
# 1. Read bronze pharmacy
# -------------------------
df = spark.table(bronze_tbl)

# -------------------------
# 2. Select and rename needed columns
# -------------------------
df = df.select(
    "pharmacy_id",
    "pharmacy_name",
    F.col("address_raw").alias("address"),
    F.col("city_raw").alias("city"),
    F.col("county_raw").alias("county"),
    F.col("state_raw").alias("state"),
    F.col("latitude_raw").alias("latitude"),
    F.col("longitude_raw").alias("longitude"),
    "ingestion_timestamp",
    "source_system"
)

# -------------------------
# 3. Basic cleaning
# -------------------------
df = (
    df.withColumn("pharmacy_id", F.trim(F.col("pharmacy_id")))
      .withColumn("pharmacy_name", F.initcap(F.trim(F.col("pharmacy_name"))))
      .withColumn("address", F.trim(F.col("address")))
      .withColumn("city", F.initcap(F.trim(F.col("city"))))
      .withColumn("county", F.initcap(F.trim(F.col("county"))))
      .withColumn("state", F.upper(F.trim(F.col("state"))))
      .withColumn("source_system", F.upper(F.trim(F.col("source_system"))))
)

# -------------------------
# 4. Safe coordinate parsing
#    Handles values like:
#    44.9778N, 93.2650W, etc.
# -------------------------
df = (
    df.withColumn(
        "latitude",
        F.expr("try_cast(regexp_replace(latitude, '[^0-9.-]', '') AS double)")
    )
    .withColumn(
        "longitude",
        F.expr("try_cast(regexp_replace(longitude, '[^0-9.-]', '') AS double)")
    )
)

# -------------------------
# 5. Fix longitude sign for W values if needed
# -------------------------
df = df.withColumn(
    "longitude",
    F.when(
        F.col("longitude").isNotNull() &
        (F.upper(F.col("state")).isin(
            "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA",
            "KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ",
            "NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT",
            "VA","WA","WV","WI","WY","DC"
        )) &
        (F.col("longitude") > 0),
        -F.col("longitude")
    ).otherwise(F.col("longitude"))
)

# -------------------------
# 6. Remove rows with null PK
# -------------------------
df = df.filter(F.col("pharmacy_id").isNotNull() & (F.col("pharmacy_id") != ""))

# -------------------------
# 7. Read geography for region lookup
# -------------------------
geo_df = (
    spark.table(geo_tbl)
    .select(
        F.initcap(F.trim(F.col("city"))).alias("geo_city"),
        F.initcap(F.trim(F.col("county"))).alias("geo_county"),
        F.upper(F.trim(F.col("state"))).alias("geo_state"),
        F.trim(F.col("geo_id")).alias("geo_id"),
        F.trim(F.col("region")).alias("region")
    )
)

# -------------------------
# 8. Join pharmacy with geography
#    to get geo_id + region
# -------------------------
df = (
    df.join(
        geo_df,
        (df.city == geo_df.geo_city) &
        (df.county == geo_df.geo_county) &
        (df.state == geo_df.geo_state),
        "left"
    )
    .select(
        df.pharmacy_id,
        df.pharmacy_name,
        df.address,
        df.city,
        df.county,
        df.state,
        F.col("region"),
        F.col("geo_id"),
        df.latitude,
        df.longitude,
        df.ingestion_timestamp,
        df.source_system
    )
)

# -------------------------
# 9. Deduplication
#    Keep latest record per pharmacy_id
# -------------------------
window_spec = Window.partitionBy("pharmacy_id").orderBy(F.col("ingestion_timestamp").desc())

df = (
    df.withColumn("row_num", F.row_number().over(window_spec))
      .filter(F.col("row_num") == 1)
      .drop("row_num")
)

# -------------------------
# 10. Write to Silver
# -------------------------
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_tbl)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# -------------------------
# Tables
# -------------------------
bronze_tbl = "capstone.bronze.pharmacies"
geo_tbl = "capstone.silver.geography"
silver_tbl = "capstone.silver.pharmacy"

# -------------------------
# 1. Read bronze pharmacy
# -------------------------
df = spark.table(bronze_tbl)

# -------------------------
# 2. Select and rename needed columns
# -------------------------
df = df.select(
    "pharmacy_id",
    "pharmacy_name",
    F.col("address_raw").alias("address"),
    F.col("city_raw").alias("city"),
    F.col("county_raw").alias("county"),
    F.col("state_raw").alias("state"),
    F.col("latitude_raw").alias("latitude"),
    F.col("longitude_raw").alias("longitude"),
    "ingestion_timestamp",
    "source_system"
)

# -------------------------
# 3. Basic cleaning
# -------------------------
df = (
    df.withColumn("pharmacy_id", F.trim(F.col("pharmacy_id")))
      .withColumn("pharmacy_name", F.initcap(F.trim(F.col("pharmacy_name"))))
      .withColumn("address", F.trim(F.col("address")))
      .withColumn("city", F.initcap(F.trim(F.col("city"))))
      .withColumn("county", F.initcap(F.trim(F.col("county"))))
      .withColumn("state", F.upper(F.trim(F.col("state"))))
      .withColumn("state", F.regexp_replace(F.col("state"), r"[^A-Z ]", ""))  # remove dots and symbols
      .withColumn("state", F.regexp_replace(F.col("state"), r"\s+", " "))     # normalize spaces
      .withColumn("source_system", F.upper(F.trim(F.col("source_system"))))
)

# -------------------------
# 4. Standardize state to 2-letter USPS code
# -------------------------
state_map = {
    "ALABAMA": "AL", "ALA": "AL", "AL": "AL",
    "ALASKA": "AK", "ALAS": "AK", "AK": "AK",
    "ARIZONA": "AZ", "ARIZ": "AZ", "AZ": "AZ",
    "ARKANSAS": "AR", "ARK": "AR", "AR": "AR",
    "CALIFORNIA": "CA", "CALIF": "CA", "CA": "CA",
    "COLORADO": "CO", "COLO": "CO", "CO": "CO",
    "CONNECTICUT": "CT", "CONN": "CT", "CT": "CT",
    "DELAWARE": "DE", "DEL": "DE", "DE": "DE",
    "FLORIDA": "FL", "FLA": "FL", "FL": "FL",
    "GEORGIA": "GA", "GA": "GA",
    "HAWAII": "HI", "HI": "HI",
    "IDAHO": "ID", "ID": "ID",
    "ILLINOIS": "IL", "ILL": "IL", "IL": "IL",
    "INDIANA": "IN", "IND": "IN", "IN": "IN",
    "IOWA": "IA", "IA": "IA",
    "KANSAS": "KS", "KAN": "KS", "KS": "KS",
    "KENTUCKY": "KY", "KY": "KY",
    "LOUISIANA": "LA", "LA": "LA",
    "MAINE": "ME", "ME": "ME",
    "MARYLAND": "MD", "MD": "MD",
    "MASSACHUSETTS": "MA", "MASS": "MA", "MA": "MA",
    "MICHIGAN": "MI", "MICH": "MI", "MI": "MI",
    "MINNESOTA": "MN", "MINN": "MN", "MN": "MN",
    "MISSISSIPPI": "MS", "MISS": "MS", "MS": "MS",
    "MISSOURI": "MO", "MO": "MO",
    "MONTANA": "MT", "MONT": "MT", "MT": "MT",
    "NEBRASKA": "NE", "NEBR": "NE", "NE": "NE",
    "NEVADA": "NV", "NEV": "NV", "NV": "NV",
    "NEW HAMPSHIRE": "NH", "NH": "NH",
    "NEW JERSEY": "NJ", "NJ": "NJ",
    "NEW MEXICO": "NM", "NM": "NM",
    "NEW YORK": "NY", "NY": "NY",
    "NORTH CAROLINA": "NC", "NC": "NC",
    "NORTH DAKOTA": "ND", "ND": "ND",
    "OHIO": "OH", "OH": "OH",
    "OKLAHOMA": "OK", "OKLA": "OK", "OK": "OK",
    "OREGON": "OR", "ORE": "OR", "OR": "OR",
    "PENNSYLVANIA": "PA", "PENN": "PA", "PA": "PA",
    "RHODE ISLAND": "RI", "RI": "RI",
    "SOUTH CAROLINA": "SC", "SC": "SC",
    "SOUTH DAKOTA": "SD", "SD": "SD",
    "TENNESSEE": "TN", "TENN": "TN", "TN": "TN",
    "TEXAS": "TX", "TEX": "TX", "TX": "TX",
    "UTAH": "UT", "UT": "UT",
    "VERMONT": "VT", "VT": "VT",
    "VIRGINIA": "VA", "VA": "VA",
    "WASHINGTON": "WA", "WASH": "WA", "WA": "WA",
    "WEST VIRGINIA": "WV", "WV": "WV",
    "WISCONSIN": "WI", "WIS": "WI", "WI": "WI",
    "WYOMING": "WY", "WYO": "WY", "WY": "WY",
    "DISTRICT OF COLUMBIA": "DC", "DC": "DC"
}

mapping_expr = F.create_map([F.lit(x) for kv in state_map.items() for x in kv])

df = df.withColumn(
    "state",
    F.coalesce(mapping_expr[F.col("state")], F.col("state"))
)

# -------------------------
# 5. Safe coordinate parsing
# -------------------------
df = (
    df.withColumn(
        "latitude",
        F.expr("try_cast(regexp_replace(latitude, '[^0-9.-]', '') AS double)")
    )
    .withColumn(
        "longitude",
        F.expr("try_cast(regexp_replace(longitude, '[^0-9.-]', '') AS double)")
    )
)

# -------------------------
# 6. Fix longitude sign for western hemisphere
# -------------------------
df = df.withColumn(
    "longitude",
    F.when(
        F.col("longitude").isNotNull() & (F.col("longitude") > 0),
        -F.col("longitude")
    ).otherwise(F.col("longitude"))
)

# -------------------------
# 7. Remove rows with null PK
# -------------------------
df = df.filter(F.col("pharmacy_id").isNotNull() & (F.col("pharmacy_id") != ""))

# -------------------------
# 8. Read geography and normalize join keys
# -------------------------
geo_df = (
    spark.table(geo_tbl)
    .select(
        F.initcap(F.trim(F.col("city"))).alias("geo_city"),
        F.initcap(F.trim(F.col("county"))).alias("geo_county"),
        F.upper(F.trim(F.col("state"))).alias("geo_state"),
        F.trim(F.col("geo_id")).alias("geo_id"),
        F.trim(F.col("region")).alias("region")
    )
    .withColumn("geo_state", F.regexp_replace(F.col("geo_state"), r"[^A-Z ]", ""))
    .withColumn("geo_state", F.regexp_replace(F.col("geo_state"), r"\s+", " "))
    .withColumn("geo_state", F.coalesce(mapping_expr[F.col("geo_state")], F.col("geo_state")))
)

# -------------------------
# 9. Join pharmacy with geography
# -------------------------
df = (
    df.join(
        geo_df,
        (df.city == geo_df.geo_city) &
        (df.county == geo_df.geo_county) &
        (df.state == geo_df.geo_state),
        "left"
    )
    .select(
        df.pharmacy_id,
        df.pharmacy_name,
        df.address,
        df.city,
        df.county,
        df.state,
        F.col("region"),
        F.col("geo_id"),
        df.latitude,
        df.longitude,
        df.ingestion_timestamp,
        df.source_system
    )
)

# -------------------------
# 10. Fill missing region from state
# -------------------------
df = df.withColumn(
    "region",
    F.when(F.col("region").isNotNull(), F.col("region"))
     .when(F.col("state").isin("CT","ME","MA","NH","RI","VT","NJ","NY","PA"), F.lit("Northeast"))
     .when(F.col("state").isin("IL","IN","MI","OH","WI","IA","KS","MN","MO","NE","ND","SD"), F.lit("Midwest"))
     .when(F.col("state").isin("DE","FL","GA","MD","NC","SC","VA","DC","WV","AL","KY","MS","TN","AR","LA","OK","TX"), F.lit("South"))
     .when(F.col("state").isin("AZ","CO","ID","MT","NV","NM","UT","WY","AK","CA","HI","OR","WA"), F.lit("West"))
     .otherwise(F.lit(None))
)

df = df.withColumn(
    "region",
    F.when(F.lower(F.trim(F.col("region"))).isin("western", "west"), "West")
     .when(F.lower(F.trim(F.col("region"))).isin("southern", "south"), "South")
     .when(F.lower(F.trim(F.col("region"))).isin("mid-west", "midwest"), "Midwest")
     .when(F.lower(F.trim(F.col("region"))) == "northeast", "Northeast")
     .otherwise(F.col("region"))
)
# -------------------------
# 11. Deduplicate by latest pharmacy_id
# -------------------------
window_spec = Window.partitionBy("pharmacy_id").orderBy(F.col("ingestion_timestamp").desc())

df = (
    df.withColumn("row_num", F.row_number().over(window_spec))
      .filter(F.col("row_num") == 1)
      .drop("row_num")
)

# -------------------------
# 12. Write to Silver
# -------------------------
df.write.format("delta").mode("overwrite").saveAsTable(silver_tbl)

In [0]:
df = df.withColumn(
    "region",
    F.when(F.col("region").isin("Western"), "West")
     .when(F.col("region").isin("Southern"), "South")
     .otherwise(F.col("region"))
)

In [0]:
%sql
SELECT
state,
region,
COUNT(*) AS total
FROM capstone.silver.pharmacy
GROUP BY state, region
ORDER BY state;